In [ ]:
!pip install --upgrade scikit-learn

In [ ]:
#Librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [ ]:
# Carga de datos, Modificar ruta segun se requiera
df = pd.read_csv('/kaggle/input/datasets/caalsial/dataset-ocurrencia/DATASET_OCURRENCIA.csv')
#Cantidad de observaciones y variables del dataset
df.shape

## 1. Análisis exploratorio

In [ ]:
#Tipo de datos del dataset
df.dtypes

In [ ]:
# Se eliminan las columnas que no se requieren para la predicción
columnas_a_eliminar = ['PAR_ID', 'FECHA_HORA', 'ANIO','LATITUD','LONGITUD','CELDA_X','CELDA_Y','TEMPERATURA_2M','PRECIPITACION','FIN_SEMANA']
df = df.drop(columns=columnas_a_eliminar)

In [ ]:
#Nuevas variables

#Hora Pico
df['ES_HORA_PICO'] = df['HORA'].apply(
    lambda x: 1 if (6 <= x <= 9) or (17 <= x <= 20) else 0
)

#Clima adverso
df['CLIMA_ADVERSO'] = ((df['LLUVIA'] > 0) & (df['NUBOSIDAD'] > 80)).astype(int)

In [ ]:
df.head(5)

In [ ]:
#Estadísticas descriptivas de las variables numéricas
df.describe()

In [ ]:
#Cantidad de datos vacíos del dataset
df.isna().sum()

In [ ]:
#Porcentaje de observaciones por ocurrencia de accidente
df['OCURRIO_ACCIDENTE'].value_counts(normalize=True)

In [ ]:
# Visualización de distribuciones de variables numéricas
numeric_cols = df.select_dtypes(include=['number']).columns.tolist() # Seleccionar columnas numéricas

# Configurar el tamaño de la figura según la cantidad de variables numéricas
num_cols = len(numeric_cols) # de variables numéricas
cols_per_row = 3
rows = (num_cols + cols_per_row - 1) // cols_per_row # Calcular número de filas necesarias

# Crear subplots para cada variable numérica
plt.figure(figsize=(cols_per_row * 5, rows * 4)) # Ajustar tamaño de la figura
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(rows, cols_per_row, i)
    sns.histplot(df[col], bins=30, kde=True, color='teal', edgecolor='black')
    plt.title(col, fontsize=11, weight='bold')
    plt.xlabel("")
    plt.ylabel("")
plt.tight_layout() # Ajustar el diseño para evitar solapamientos
plt.show()

In [ ]:
#Matriz de correlación
corr=df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', annot_kws={'size': 9})
plt.title("Mapa de calor de correlación")
plt.tight_layout()
plt.show()

In [ ]:
accidentes_por_mes = df.groupby('MES')['OCURRIO_ACCIDENTE'].sum()

# Crear la figura
plt.figure(figsize=(10, 6))
accidentes_por_mes.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Cantidad de Accidentes por Mes', fontsize=14, pad=15)
plt.xlabel('Mes', fontsize=12)
plt.ylabel('Total de Accidentes', fontsize=12)
plt.xticks(rotation=0) 
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
accidentes_por_dia = df.groupby('DIA_SEMANA')['OCURRIO_ACCIDENTE'].sum()

# Crear la figura
plt.figure(figsize=(10, 6))
accidentes_por_dia.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Cantidad de Accidentes por Dia', fontsize=14, pad=15)
plt.xlabel('Dia', fontsize=12)
plt.ylabel('Total de Accidentes', fontsize=12)
plt.xticks(rotation=0) 
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.stripplot(x='OCURRIO_ACCIDENTE', y='LLUVIA', data=df, 
              jitter=True, alpha=0.3, color='teal')

plt.title('Dispersión de Lluvia vs. Ocurrencia de Accidentes', fontsize=14)
plt.xlabel('Ocurrió Accidente', fontsize=12)
plt.ylabel('Cantidad de Lluvia', fontsize=12)

plt.show()

## 2. Conjunto de datos de entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

# Se separan las variables
X = df.drop(columns=['OCURRIO_ACCIDENTE'])
y = df['OCURRIO_ACCIDENTE'] 

#  Se dividen los datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20,
    random_state=42
)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

In [ ]:
# 4. CREAR 'ACCIDENTALIDAD_CELDA' (Target Encoding Seguro)
# Calculamos el porcentaje histórico SOLO con los datos de entrenamiento
diccionario_accidentalidad = y_train.groupby(X_train['CELDA_ID']).mean()

# Mapeamos este porcentaje a Train y Test
X_train['ACCIDENTALIDAD_CELDA'] = X_train['CELDA_ID'].map(diccionario_accidentalidad)
X_test['ACCIDENTALIDAD_CELDA'] = X_test['CELDA_ID'].map(diccionario_accidentalidad)

# Llenar nulos (por si en Test hay una celda nueva) con el promedio global de la ciudad
promedio_global = y_train.mean()
X_train['ACCIDENTALIDAD_CELDA'] = X_train['ACCIDENTALIDAD_CELDA'].fillna(promedio_global)
X_test['ACCIDENTALIDAD_CELDA'] = X_test['ACCIDENTALIDAD_CELDA'].fillna(promedio_global)

# Finalmente, borramos el CELDA_ID de texto porque el modelo ya no lo necesita
X_train = X_train.drop(columns=['CELDA_ID'])
X_test = X_test.drop(columns=['CELDA_ID'])

print("¡Transformación completada con éxito!")
print(f"Dimensiones de X_train: {X_train.shape}")

## 3. Entrenamiento y prueba de modelos de clasificación

### 3.1 Regresión logística

In [ ]:
#Librerias de sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score,precision_score, recall_score, log_loss)

In [ ]:
#Se escalan los dato
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Se crea el modelo base de regresión logística
log_reg_base= LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
#Calibración de hiperparámetros
parametros_lr = {
'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0.0, 1.0, 0.25, 0.5, 0.75], 
    'solver': ['saga']
}

#Grid search para encontrar el mejor modelo
grid_search_lr = GridSearchCV(
    estimator=log_reg_base,
    param_grid=parametros_lr,
    cv=5,                 # Validación cruzada de 5 particiones
    scoring='roc_auc',
    n_jobs=-1
)

#Entrenamiento de los modelos

inicio_tiempo = time.time()

grid_search_lr.fit(X_train_scaled, y_train)

fin_tiempo = time.time()
tiempo_total = fin_tiempo - inicio_tiempo
minutos = int(tiempo_total // 60)
segundos = int(tiempo_total % 60)

print(f"Duración del entrenamiento {minutos} minutos y {segundos} segundos")

mejor_lr = grid_search_lr.best_estimator_

print(f"Mejores parámetros encontrados: {grid_search_lr.best_params_}\n")

In [ ]:
#Predicciones
y_pred_lr = mejor_lr.predict(X_test_scaled)
y_prob_lr = mejor_lr.predict_proba(X_test_scaled)[:, 1]

In [ ]:
#Se calculan las métricas
metricas = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
    "F1 Score": f1_score(y_test, y_pred_lr),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_lr),
    "Precision": precision_score(y_test, y_pred_lr),
    "Recall (Sensibilidad)": recall_score(y_test, y_pred_lr),
    "Log Loss": log_loss(y_test, y_prob_lr)
}

print("--- Resultados del Modelo: Regresión Logística Calibrada ---")
for metrica, valor in metricas.items():
    print(f"{metrica:<22}: {valor:.4f}")

### 3.2 Random Forest Classifier

In [ ]:
#Librerias de sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score,precision_score, recall_score, log_loss)
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Se crea el modelo base de RF Classifier
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

In [ ]:
#Calibración de hiperparámetros
parametros_rf = {
    'n_estimators': [50, 100, 150],      
    'max_depth': [5, 10],          
    'min_samples_split': [2, 5] 
}

#Grid search para encontrar el mejor modelo
grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=parametros_rf,
    cv=5,                 # Validación cruzada de 5 particiones
    scoring='roc_auc',
    n_jobs=-1
)

#Entrenamiento de los modelos

inicio_tiempo = time.time()

grid_search_rf.fit(X_train, y_train)

fin_tiempo = time.time()
tiempo_total = fin_tiempo - inicio_tiempo
minutos = int(tiempo_total // 60)
segundos = int(tiempo_total % 60)

print(f"Duración del entrenamiento {minutos} minutos y {segundos} segundos")

mejor_rf = grid_search_rf.best_estimator_

print(f"Mejores parámetros encontrados: {grid_search_rf.best_params_}\n")

In [ ]:
# Se encuentran las variables de mayor importancia
importancias = mejor_rf.feature_importances_

df_importancias = pd.DataFrame({'Variable': X_train.columns, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

df_importancias

In [ ]:
#Predicciones
y_pred_lr_rf = mejor_rf.predict(X_test)
y_prob_lr_rf = mejor_rf.predict_proba(X_test)[:, 1]

In [ ]:
#Se calculan las métricas
metricas_rf = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr_rf),
    "F1 Score": f1_score(y_test, y_pred_lr_rf),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_lr_rf),
    "Precision": precision_score(y_test, y_pred_lr_rf),
    "Recall (Sensibilidad)": recall_score(y_test, y_pred_lr_rf),
    "Log Loss": log_loss(y_test, y_prob_lr_rf)
}

print("--- Resultados de Random Forest (Optimizado, Sin Calibrar) ---")
for metrica, valor in metricas_rf.items():
    print(f"{metrica:<22}: {valor:.4f}")

### 3.3 XGBoost Classifier

In [ ]:
!pip install --upgrade xgboost

In [ ]:
# Librerías
from xgboost import XGBClassifier

In [ ]:
#Modelo baso de XGB Classifier
xgb_base = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', device='cuda', tree_method='hist')

In [ ]:
# Calibración de hiperparámetros
parametros_xgb = {
    'n_estimators': [50, 100, 150],      
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2] 
}

#Grid search para encontrar el mejor modelo
grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=parametros_xgb,
    cv=5,                 # Validación cruzada de 5 particiones
    scoring='roc_auc',
    n_jobs=-1
)

# Entrenamiento de los modelos
inicio_tiempo = time.time()

grid_search_xgb.fit(X_train, y_train)

fin_tiempo = time.time()
tiempo_total = fin_tiempo - inicio_tiempo
minutos = int(tiempo_total // 60)
segundos = int(tiempo_total % 60)

print(f"Duración del entrenamiento {minutos} minutos y {segundos} segundos")

mejor_xgb = grid_search_xgb.best_estimator_

print(f"Mejores parámetros encontrados: {grid_search_xgb.best_params_}\n")

In [ ]:
#Predicciones
y_pred_xgb = mejor_xgb.predict(X_test)
y_prob_xgb = mejor_xgb.predict_proba(X_test)[:, 1]

In [ ]:
#Metricas
metricas_xgb = {
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb),
    "F1 Score": f1_score(y_test, y_pred_xgb),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_xgb),
    "Precision": precision_score(y_test, y_pred_xgb),
    "Recall (Sensibilidad)": recall_score(y_test, y_pred_xgb),
    "Log Loss": log_loss(y_test, y_prob_xgb)
}

print("--- Resultados de XGBoost ---")
for metrica, valor in metricas_xgb.items():
    print(f"{metrica:<22}: {valor:.4f}")

### 3.5 Comparación de modelos

In [ ]:
df_comparacion = pd.DataFrame({
    'Regresión Logística': metricas,
    'Random Forest': metricas_rf,
    'XGBoost': metricas_xgb
})


print("--- Comparación de Modelos ---")
display(df_comparacion.round(4))